In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

In [ ]:
# Environment & Paths
load_dotenv()

BASE_DIR      = Path().resolve().parent
RAW_DATA_DIR  = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_FILE_NAME = os.getenv("RAW_FILE_NAME")
RAW_DATA      = RAW_DATA_DIR / RAW_FILE_NAME

In [ ]:
# Load Data and view first 5 rows
df = pd.read_csv(RAW_DATA)
print(df.head())
print(df.shape)

In [ ]:
## Create Subsets
# clients
target_cols_clients = ["client_id", "age", "job", "marital", "education", "credit_default","mortgage"]
df_clients = df[target_cols_clients]

# campaigns
target_cols_campaigns = ["client_id", "number_contacts", "contact_duration", "previous_campaign_contacts", "previous_outcome","campaign_outcome"]
df_campaigns = df[target_cols_campaigns]

# economics
target_cols_economics = ["client_id", "cons_price_idx","euribor_three_months"]
df_economics = df[target_cols_economics]

In [ ]:
## Clean and transform data for clients
# For "job" column: change "." to "_"
df_clients["job"] = df_clients["job"].str.replace(".","_", regex=False)

# For "education" column: change "." to "_" and "unknown" to np.NaN
df_clients["education"] = (df_clients["education"].str.replace(".", "_", regex=False).replace("unknown", np.nan))

# Convert yes/no columns to boolean
bool_columns = ["credit_default", "mortgage"]

for col in bool_columns:
    df_clients[col] = (df_clients[col]
        .map({"yes": True, "no": False, "unknown": False})
        .astype(bool)
    )

print(df_clients.head())

In [ ]:
## Clean and transform data for campaigns
# Create a new colum for "year" in the main data frame
df["year"] = 2022

# Create a new column called "last_contact_date" using the columns "day" and "month" and "year" for the campaign data frame
df_campaigns["last_contact_date"] = pd.to_datetime(df["year"].astype(str) + "-" + df["month"] + "-" + df["day"].astype(str), format="%Y-%b-%d")

# Convert outcome columns to boolean
outcome_map = {
    "campaign_outcome":  {"yes": True, "no": False},
    "previous_outcome":  {"success": True, "failure": False, "nonexistent": False}
}

for col, mapping in outcome_map.items():
    df_campaigns[col] = df_campaigns[col].map(mapping).astype(bool)

print(df_campaigns.head())

In [ ]:
# Print value counts for target columns
for col in ["credit_default", "mortgage", "previous_outcome", "campaign_outcome"]:
    print(col)
    print("--------------")
    print(df[col].value_counts())

In [ ]:
# Export Processed Data
df_clients.to_csv(PROCESSED_DIR / "client.csv", index=False)
df_campaigns.to_csv(PROCESSED_DIR / "campaign.csv", index=False)
df_economics.to_csv(PROCESSED_DIR / "economics.csv", index=False)

print("\nFiles saved to data/processed/")